# Insurance - Deploy Fraud Detection Model to Azure ML

Deploys the insurance fraud detection model to an Azure ML Managed Online Endpoint,
reusing the existing YAML config files under `insurance/deploy/`.

Set the `ENVIRONMENT` widget (or the `ENVIRONMENT` env var) to control which
configuration is used:

| Setting | Staging | Production |
|---|---|---|
| Endpoint suffix | `-staging` | `-prod` |
| Instance count | 1 | 2 |

**Prerequisites:**
- `azure-ai-ml` and `azure-identity` installed on the cluster
- Databricks secret scope `azure-ml` with keys: `subscription-id`, `resource-group`, `workspace-name`
- Model artefacts present in `insurance/artefacts/` (run `02_train_model` first)

**Steps:**
1. Resolve environment configuration and Azure ML workspace coordinates
2. Connect to Azure ML workspace
3. Create or update the managed online endpoint
4. Create or update the deployment
5. Route 100% traffic to the deployment
6. Poll deployment status until succeeded
7. Test the endpoint with a sample scoring request
8. Print summary

In [ ]:
%run ../_setup

In [ ]:
import json
import time
from pathlib import Path

import requests
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
)
from azure.identity import DefaultAzureCredential

In [ ]:
# ---------------------------------------------------------------------------
# Environment configuration
# ---------------------------------------------------------------------------
INDUSTRY = "insurance"

ENV_PROFILES = {
    "staging": {
        "endpoint_suffix": "-staging",
        "instance_count": 1,
    },
    "production": {
        "endpoint_suffix": "-prod",
        "instance_count": 2,
    },
}

# Resolve environment: Databricks widget > env var > default to staging
try:
    ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT").lower()  # noqa: F821
except Exception:
    ENVIRONMENT = os.environ.get("ENVIRONMENT", "staging").lower()

if ENVIRONMENT not in ENV_PROFILES:
    raise ValueError(
        f"Unknown ENVIRONMENT '{ENVIRONMENT}'. Must be one of: {list(ENV_PROFILES.keys())}"
    )

env = ENV_PROFILES[ENVIRONMENT]

ENDPOINT_BASE_NAME = "insurance-fraud-detection-endpoint"
DEPLOYMENT_BASE_NAME = "insurance-fraud-detection-v1"
ENDPOINT_NAME = f"{ENDPOINT_BASE_NAME}{env['endpoint_suffix']}"
DEPLOYMENT_NAME = f"{DEPLOYMENT_BASE_NAME}{env['endpoint_suffix']}"

# ---------------------------------------------------------------------------
# Azure ML workspace coordinates
# ---------------------------------------------------------------------------
def _get_secret(key, env_var):
    """Retrieve a secret from Databricks or fall back to an env var."""
    try:
        return dbutils.secrets.get(scope="azure-ml", key=key)  # noqa: F821
    except Exception:
        return os.environ[env_var]

SUBSCRIPTION_ID = _get_secret("subscription-id", "AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = _get_secret("resource-group", "AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = _get_secret("workspace-name", "AZURE_ML_WORKSPACE_NAME")

DEPLOY_DIR = Path(_repo_root) / INDUSTRY / "deploy"

print(f"Environment     : {ENVIRONMENT}")
print(f"Endpoint name   : {ENDPOINT_NAME}")
print(f"Deployment name : {DEPLOYMENT_NAME}")
print(f"Instance count  : {env['instance_count']}")
print(f"Subscription    : {SUBSCRIPTION_ID[:8]}...")
print(f"Resource group  : {RESOURCE_GROUP}")
print(f"Workspace       : {WORKSPACE_NAME}")
print(f"Deploy dir      : {DEPLOY_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Connect to Azure ML workspace
# ---------------------------------------------------------------------------
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to Azure ML workspace: {ml_client.workspace_name}")

In [ ]:
# ---------------------------------------------------------------------------
# Create or update the managed online endpoint
# ---------------------------------------------------------------------------
endpoint = ManagedOnlineEndpoint.load(str(DEPLOY_DIR / "endpoint.yml"))
endpoint.name = ENDPOINT_NAME

print(f"Creating/updating endpoint: {endpoint.name} ...")
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint ready: {endpoint.name}")

In [ ]:
# ---------------------------------------------------------------------------
# Create or update the deployment
# ---------------------------------------------------------------------------
deployment = ManagedOnlineDeployment.load(str(DEPLOY_DIR / "deployment.yml"))
deployment.name = DEPLOYMENT_NAME
deployment.endpoint_name = ENDPOINT_NAME
deployment.instance_count = env["instance_count"]

print(f"Creating/updating deployment: {deployment.name} ...")
print(f"  Instance type  : {deployment.instance_type}")
print(f"  Instance count : {deployment.instance_count}")
ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment ready: {deployment.name}")

In [ ]:
# ---------------------------------------------------------------------------
# Route 100% traffic to the new deployment
# ---------------------------------------------------------------------------
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic set to 100% for deployment: {DEPLOYMENT_NAME}")

In [ ]:
# ---------------------------------------------------------------------------
# Poll deployment status until Succeeded
# ---------------------------------------------------------------------------
MAX_WAIT_SECONDS = 600
POLL_INTERVAL = 15
elapsed = 0

while elapsed < MAX_WAIT_SECONDS:
    dep_state = ml_client.online_deployments.get(
        name=DEPLOYMENT_NAME, endpoint_name=ENDPOINT_NAME
    )
    status = dep_state.provisioning_state
    print(f"  [{elapsed:>3d}s] Deployment status: {status}")
    if status == "Succeeded":
        break
    if status == "Failed":
        raise RuntimeError(
            f"Deployment '{DEPLOYMENT_NAME}' failed. "
            "Check Azure ML Studio for details."
        )
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise TimeoutError(
        f"Deployment '{DEPLOYMENT_NAME}' did not reach 'Succeeded' "
        f"within {MAX_WAIT_SECONDS} seconds."
    )

print(f"Deployment '{DEPLOYMENT_NAME}' is live.")

In [ ]:
# ---------------------------------------------------------------------------
# Test the endpoint with a sample scoring request
# ---------------------------------------------------------------------------
scoring_uri = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
api_key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key

sample_payload = {
    "data": [
        {
            "claim_amount": 85000,
            "policy_tenure_months": 36,
            "customer_age": 42,
            "num_prior_claims": 2,
            "premium_amount": 4500,
            "days_to_report": 5,
            "witness_present": 1,
            "police_report_filed": 0,
            "vehicle_age_years": 4,
            "incident_type": "Collision",
            "province": "Western Cape",
        }
    ]
}

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}",
}

resp = requests.post(scoring_uri, headers=headers, json=sample_payload)
resp.raise_for_status()

result = resp.json()
print("Sample request payload:")
print(json.dumps(sample_payload, indent=2))
print("\nEndpoint response:")
print(json.dumps(result, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
studio_url = (
    f"https://ml.azure.com/endpoints/realtime/{ENDPOINT_NAME}"
    f"?wsid=/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{WORKSPACE_NAME}"
)

print(f"\n{'='*60}")
print(f"  Insurance Fraud Detection - Azure ML Deployment")
print(f"{'='*60}")
print(f"  Environment     : {ENVIRONMENT}")
print(f"  Endpoint name   : {ENDPOINT_NAME}")
print(f"  Deployment name : {DEPLOYMENT_NAME}")
print(f"  Instance count  : {env['instance_count']}")
print(f"  Scoring URI     : {scoring_uri}")
print(f"{'='*60}")
print(f"\nView in Azure ML Studio:")
print(f"  {studio_url}")